# MLPC 2026 — Challenge Section 2: Classical Classifier Swap

**Team: Validated Encoders — Amr & Ibrahim**

Goal: improve the provided SED baseline by **replacing the per-class DecisionTree
with HistGradientBoosting**, while keeping every other part of the pipeline
identical (segmentation, half-second-segment discarding, interval merging,
Segment-based Macro F1 evaluation).

This is a *controlled* swap: both classifiers are trained on the **same**
`X_train / Y_train` and evaluated on the **same** local-validation split, so any
difference in macro-F1 is attributable to the classifier alone.

**Guardrails honored:** model selection here uses the **local validation set only**.
The non-hidden test set is reserved for a single final evaluation after selection.
No threshold tuning here (default 0.5) — that is the separate next sub-task.

## Cell 1 — Setup, constants, paths

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict

# --- Feature + class definitions (identical to the baseline) ---
FEATURE_NAMES = [
    "zcr_mean",        "zcr_std",        "zcr_min",        "zcr_max",
    "melspect_mean",   "melspect_std",   "melspect_min",   "melspect_max",
    "mfcc_mean",       "mfcc_std",       "mfcc_min",       "mfcc_max",
    "mfcc_d_mean",     "mfcc_d_std",     "mfcc_d_min",     "mfcc_d_max",
    "mfcc_d2_mean",    "mfcc_d2_std",    "mfcc_d2_min",    "mfcc_d2_max",
    "flux_mean",       "flux_std",       "flux_min",       "flux_max",
    "flatness_mean",   "flatness_std",   "flatness_min",   "flatness_max",
    "centroid_mean",   "centroid_std",   "centroid_min",   "centroid_max",
    "bandwidth_mean",  "bandwidth_std",  "bandwidth_min",  "bandwidth_max",
    "contrast_mean",   "contrast_std",   "contrast_min",   "contrast_max",
    "rolloff_low_mean",  "rolloff_low_std",  "rolloff_low_min",  "rolloff_low_max",
    "rolloff_high_mean", "rolloff_high_std", "rolloff_high_min", "rolloff_high_max",
    "energy_mean",     "energy_std",     "energy_min",     "energy_max",
    "power_mean",      "power_std",      "power_min",      "power_max",
]

CLASS_NAMES = [
    "bell_ringing", "coffee_machine", "cutlery_dishes", "door_open_close",
    "footsteps", "keyboard_typing", "keychain", "light_switch", "microwave",
    "phone_ringing", "running_water", "toilet_flushing", "vacuum_cleaner",
    "wardrobe_drawer_open_close", "window_open_close",
]

SEGMENT_LENGTH = 1.0   # each feature vector covers a 1-second window
HOP_SIZE       = 0.5   # segments extracted with 50% overlap

# --- Paths ---
PATH_TO_DATASET = "/Users/amrabdullah/AI_2026ss/mlpc/phases/04_challenge/data"
PATH_TRAIN = os.path.join(PATH_TO_DATASET, "train")
PATH_VAL   = os.path.join(PATH_TO_DATASET, "validation")
PATH_TEST  = os.path.join(PATH_TO_DATASET, "test")

# evaluate.py lives in the baseline/ folder — import the EXACT scoring logic.
BASELINE_DIR = "/Users/amrabdullah/AI_2026ss/mlpc/phases/04_challenge/baseline"
sys.path.insert(0, BASELINE_DIR)
from evaluate import (
    aggregate_ground_truth_annotations,
    build_segment_frame_from_intervals,
    calculate_f1_score,
)

for path in [PATH_TRAIN, PATH_VAL, PATH_TEST, os.path.join(BASELINE_DIR, "evaluate.py")]:
    assert os.path.exists(path), f"Path not found: {path}"
print("Setup OK. Features:", len(FEATURE_NAMES), "| Classes:", len(CLASS_NAMES))

## Cell 2 — Data-loading helpers (copied verbatim from the baseline)

These are reused unchanged so the feature matrix and the label binarization
(`>0` overlap, then majority vote across annotators) are *identical* to the
baseline. Only the classifier downstream will differ.

In [ ]:
import numpy as np
from typing import List, Tuple

def build_feature_matrix(data: dict) -> np.ndarray:
    """Concatenate all named features from a loaded .npz dict into one matrix."""
    arrays = []
    for feat_name in FEATURE_NAMES:
        feat = data[feat_name]
        if feat.ndim == 1:
            feat = feat[:, np.newaxis]
        arrays.append(feat.astype(np.float32))
    return np.concatenate(arrays, axis=1)

def get_segment_labels(data: dict) -> np.ndarray:
    """Binary multilabel targets: overlap>0 per annotator, then majority vote."""
    annotations = data["annotations"]            # (N, C, A)
    binary = (annotations > 0).astype(int)       # (N, C, A)
    votes = binary.sum(axis=2)                   # (N, C)
    n_annotators = binary.shape[2]
    y = (votes > (n_annotators // 2)).astype(int)
    return y

def load_all_segments(file_list: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    """Load features and labels from a list of .npz files and stack them."""
    X_list, Y_list = [], []
    for filepath in file_list:
        data = dict(np.load(filepath, allow_pickle=True))
        X_list.append(build_feature_matrix(data))
        Y_list.append(get_segment_labels(data))
    return np.vstack(X_list), np.vstack(Y_list)

print("Data helpers defined.")

## Cell 3 — Load file lists and the seed=42 validation split

Identical split to the baseline (`np.random.default_rng(seed=42)`), so
`our_val_files` and `our_test_files` contain exactly the same recordings as in
the baseline notebook. We will only use `our_val_files` for model selection.

In [ ]:
import glob, os
import numpy as np

train_files = sorted(glob.glob(os.path.join(PATH_TRAIN, "audio_features", "*.npz")))
val_files   = sorted(glob.glob(os.path.join(PATH_VAL,   "audio_features", "*.npz")))
test_files  = sorted(glob.glob(os.path.join(PATH_TEST,  "audio_features", "*.npz")))

rng = np.random.default_rng(seed=42)
val_files_shuffled = rng.permutation(val_files).tolist()
n_val = len(val_files_shuffled) // 2
our_val_files  = val_files_shuffled[:n_val]    # local validation (model selection)
our_test_files = val_files_shuffled[n_val:]    # non-hidden test (single final eval — NOT used here)

print(f"Train recordings:        {len(train_files)}")
print(f"Local validation set:    {len(our_val_files)}")
print(f"Non-hidden test set:     {len(our_test_files)}  (reserved — not touched in this notebook)")
print(f"Hidden test set:         {len(test_files)}")

## Cell 4 — Build the training matrix (same 50k subsample as the baseline)

Same `MAX_TRAINING_SEGMENTS = 50_000` subsample with the same RNG, so both
classifiers see the same training data. The class-activity distribution printed
here documents the imbalance that the transient classes suffer from.

In [ ]:
import numpy as np

MAX_TRAINING_SEGMENTS = 50_000

print(f"Loading features from {len(train_files)} training recordings...")
X_train, Y_train = load_all_segments(train_files)
print(f"Loaded {X_train.shape[0]} segments x {X_train.shape[1]} features.")

if X_train.shape[0] > MAX_TRAINING_SEGMENTS:
    idx = rng.choice(X_train.shape[0], size=MAX_TRAINING_SEGMENTS, replace=False)
    X_train, Y_train = X_train[idx], Y_train[idx]
    print(f"Subsampled to {X_train.shape[0]} segments.")

print(f"\nFinal training set: {X_train.shape[0]} segments x {X_train.shape[1]} features, {Y_train.shape[1]} classes.\n")
print("Class activity (fraction of segments active):")
for i, cls in enumerate(CLASS_NAMES):
    frac = Y_train[:, i].mean()
    print(f"  {cls:<30} {frac:.3f}  {'#' * int(frac*40)}")

## Cell 5 — SED inference + evaluation helpers (copied verbatim from the baseline)

`run_sed_inference` discards half-second segments and calls `classifier.predict`
(hard 0/1 at the default 0.5 threshold). `evaluate_split` calls the provided
`evaluate.py` to compute the exact Segment-based Macro F1. Unchanged from the
baseline so the swap is the only difference.

In [ ]:
import os
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

def run_sed_inference(filepath, classifier):
    data = dict(np.load(filepath, allow_pickle=True))
    start_times_all = data["start_time"]
    X_all = build_feature_matrix(data)
    pred_all = classifier.predict(X_all)
    whole_second_mask = np.isclose(start_times_all % 1.0, 0.0)
    predictions = pred_all[whole_second_mask]
    start_times = start_times_all[whole_second_mask]
    filename = os.path.basename(filepath).replace(".npz", ".wav")
    return predictions, start_times, filename

def predictions_to_intervals(predictions, start_times, filename):
    rows = []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_preds = predictions[:, cls_idx]
        in_event, onset = False, None
        for t, pred in zip(start_times, cls_preds):
            if pred == 1 and not in_event:
                onset, in_event = float(t), True
            elif pred == 0 and in_event:
                rows.append({"filename": filename, "annotation": cls_name,
                             "onset": onset, "offset": float(t)})
                in_event = False
        if in_event:
            rows.append({"filename": filename, "annotation": cls_name,
                         "onset": onset, "offset": float(start_times[-1]) + SEGMENT_LENGTH})
    return rows

def generate_predictions(file_list, classifier):
    all_rows = []
    for filepath in file_list:
        preds, times, fname = run_sed_inference(filepath, classifier)
        all_rows.extend(predictions_to_intervals(preds, times, fname))
    if not all_rows:
        return pd.DataFrame(columns=["filename", "annotation", "onset", "offset"])
    return pd.DataFrame(all_rows)

def evaluate_split(pred_df, file_list, ann_df):
    split_filenames = {os.path.basename(f).replace(".npz", ".wav") for f in file_list}
    ann_split = ann_df[ann_df["filename"].isin(split_filenames)].copy()
    gt = aggregate_ground_truth_annotations(ann_split)
    gt_segments   = build_segment_frame_from_intervals(gt,      name="ground_truth")
    pred_segments = build_segment_frame_from_intervals(pred_df, name="predictions")
    if len(pred_segments) > 0:
        pf = pred_segments.index.get_level_values("filename")
        pred_segments = pred_segments[pf.isin(split_filenames)]
    return calculate_f1_score(gt_segments, pred_segments)

print("Inference + evaluation helpers defined.")

## Cell 6 — Train both classifiers and evaluate on the LOCAL VALIDATION set

Both wrapped in `MultiOutputClassifier` (one estimator per class, 15 total),
trained on the identical `X_train / Y_train`.

- **Baseline reference:** `DecisionTreeClassifier(max_depth=20, max_features='sqrt')`
- **New:** `HistGradientBoostingClassifier` (sklearn histogram GBM)

HistGB hyperparameters are a sensible first setting (not yet tuned — tuning is a
later sub-task): `max_iter=300, learning_rate=0.1, max_depth=None,
max_leaf_nodes=31, early_stopping`. No feature scaling (histogram trees are
scale-invariant).

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

ann_df = pd.read_csv(os.path.join(PATH_VAL, "annotations.csv"))

models = {
    "DecisionTree (baseline)": MultiOutputClassifier(
        DecisionTreeClassifier(max_depth=20, max_features="sqrt", random_state=42),
        n_jobs=-1),
    "HistGradientBoosting": MultiOutputClassifier(
        HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.1, max_leaf_nodes=31,
            early_stopping=True, validation_fraction=0.1,
            random_state=42),
        n_jobs=-1),
}

results = {}
for name, clf in models.items():
    print(f"\n=== {name} ===")
    t0 = time.time()
    clf.fit(X_train, Y_train)
    print(f"  trained in {time.time()-t0:.1f}s")
    pred_val = generate_predictions(our_val_files, clf)
    macro_f1, per_class = evaluate_split(pred_val, our_val_files, ann_df)
    results[name] = {"clf": clf, "macro_f1": macro_f1, "per_class": per_class,
                     "n_intervals": len(pred_val)}
    print(f"  Local-validation Macro F1: {macro_f1:.4f}  ({len(pred_val)} intervals)")

print("\n--- Summary (local validation) ---")
for name, r in results.items():
    print(f"  {name:<28} macro-F1 = {r['macro_f1']:.4f}")

## Cell 7 — Per-class comparison

Side-by-side per-class F1 for the two classifiers on the local validation set.
Watch the transient classes (`door_open_close`, `window_open_close`,
`wardrobe_drawer_open_close`, `light_switch`) — these were the weakest in Task 4
and are where the swap is expected to matter least (1-second aggregation erases
short events regardless of classifier).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = results["DecisionTree (baseline)"]["per_class"].set_index("annotation")["f1"]
hgb = results["HistGradientBoosting"]["per_class"].set_index("annotation")["f1"]
order = CLASS_NAMES
dt_f1  = [dt.get(c, 0.0) for c in order]
hgb_f1 = [hgb.get(c, 0.0) for c in order]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(order)); w = 0.4
ax.bar(x - w/2, dt_f1,  w, label=f"DecisionTree (macro={results['DecisionTree (baseline)']['macro_f1']:.3f})", color="coral")
ax.bar(x + w/2, hgb_f1, w, label=f"HistGB (macro={results['HistGradientBoosting']['macro_f1']:.3f})", color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_ylabel("F1"); ax.set_ylim(0, 1.05)
ax.set_title("Per-class F1 — local validation — DecisionTree vs HistGradientBoosting")
ax.legend(); plt.tight_layout(); plt.show()

cmp = pd.DataFrame({"class": order, "DT_f1": dt_f1, "HistGB_f1": hgb_f1})
cmp["delta"] = cmp["HistGB_f1"] - cmp["DT_f1"]
print(cmp.to_string(index=False))
print(f"\nMacro-F1  DT={results['DecisionTree (baseline)']['macro_f1']:.4f}  "
      f"HistGB={results['HistGradientBoosting']['macro_f1']:.4f}  "
      f"delta={results['HistGradientBoosting']['macro_f1']-results['DecisionTree (baseline)']['macro_f1']:+.4f}")